Структура панели 
Распределение состояний
Матрица переходов 
Default rate 
Признаки: univariate vs риск
признак ↔ состояние
автокорреляция/устойчивость 
Train/Test drift 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path


from src.data.processed_reader import load_processed_features, load_processed_panel
from src.features.feature_contracts import FEATURE_COLS


try:
    from src.models.markov_chain import estimate_transition_matrix
except ImportError:
    estimate_transition_matrix = None

DEFAULT_STATE = 2
STATE_NAMES = {0: "current", 1: "delinquent", 2: "default"}
OUT_DIR = Path("reports/eda")

ModuleNotFoundError: No module named 'src'

: 

Описание сырых данных: 
 распределение состояний 
 динамика по месяцам 
 визуализация миграций 

In [ ]:
overview = pd.Series({
    "panel_строк": len(panel),
    "panel_клиентов": panel["client_id"].nunique(),
    "feat_строк": len(feat),
    "feat_клиентов": feat["client_id"].nunique(),
    "месяц_мин": int(feat["month"].min()),
    "месяц_макс": int(feat["month"].max()),
    "пропусков_panel": int(panel.isna().sum().sum()),
    "пропусков_feat": int(feat.isna().sum().sum()),
    "доля_next_default": float((feat["next_state"] == DEFAULT_STATE).mean()),
})
display(overview)
print(panel.dtypes)
print(panel.isnull().sum())
display(panel.describe())

In [ ]:
print("state в panel")
print(panel["state"].value_counts(normalize=True).sort_index())
print("\nnext_state")
print(feat["next_state"].value_counts(normalize=True).sort_index())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
vc = panel["state"].value_counts(normalize=True).sort_index()
axes[0].bar([STATE_NAMES.get(i, i) for i in vc.index], vc.values)
axes[0].set_title("state в panel")
vc = feat["next_state"].value_counts(normalize=True).sort_index()
axes[1].bar([STATE_NAMES.get(int(i), i) for i in vc.index], vc.values, color="#E45756")
axes[1].set_title("next_state")
plt.show()

In [ ]:
mix = (
    panel.groupby("month")["state"]
    .value_counts(normalize=True)
    .rename("share")
    .reset_index()
    .pivot(index="month", columns="state", values="share")
    .fillna(0)
)
display(mix)

fig, ax = plt.subplots(figsize=(7, 4))
for s, name in STATE_NAMES.items():
    if s in mix.columns:
        ax.plot(mix.index, mix[s], marker="o", label=name)
ax.set_title("Смесь состояний по месяцам")
ax.legend()
plt.show()

key = [c for c in ["utilization", "util_mean", "util_trend", "frac_delinquent"] if c in feat.columns]
feat[key].hist(bins=40, figsize=(12, 3))
plt.suptitle("Ключевые признаки")
plt.show()

In [ ]:
y = (feat["next_state"] == DEFAULT_STATE).astype(float)
rows = []
for c in FEATURE_COLS:
    if c not in feat.columns:
        continue
    x = pd.to_numeric(feat[c], errors="coerce")
    if x.nunique(dropna=True) < 2 or y.nunique() < 2:
        continue
    r = x.corr(y)
    if pd.notna(r):
        rows.append({"признак": c, "корр": float(r)})

corr = pd.DataFrame(rows).sort_values("корр", key=lambda s: s.abs(), ascending=False)
display(corr.head(15))

top = corr.head(12)
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top["признак"][::-1], top["корр"][::-1])
ax.set_title("corr с next_state==default")
plt.show()

In [ ]:
use = [
    c for c in FEATURE_COLS
    if c in feat.columns and pd.to_numeric(feat[c], errors="coerce").nunique(dropna=True) >= 2
]
cm = feat[use].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap="vlag", center=0, vmin=-1, vmax=1)
plt.title("Корреляции признаков")
plt.show()

In [ ]:
use = [
    c for c in FEATURE_COLS
    if c in feat.columns and pd.to_numeric(feat[c], errors="coerce").nunique(dropna=True) >= 2
]
cm = feat[use].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap="vlag", center=0, vmin=-1, vmax=1)
plt.title("Корреляции признаков")
plt.show()

In [ ]:
pairs = feat.dropna(subset=["next_state"]).copy()
pairs["state"] = pairs["state"].astype(int)
pairs["next_state"] = pairs["next_state"].astype(int)

N = pd.crosstab(pairs["state"], pairs["next_state"])
P = N.div(N.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
display(N)
display(P)

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(
    P,
    annot=True,
    fmt=".3f",
    cmap="YlOrRd",
    vmin=0,
    vmax=1,
    xticklabels=[STATE_NAMES.get(i, i) for i in P.columns],
    yticklabels=[STATE_NAMES.get(i, i) for i in P.index],
)
plt.title("Статическая матрица P")
plt.xlabel("t+1")
plt.ylabel("t")
plt.show()